In [8]:
# ============================================================
# MARK 1
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import Pinecone

from pinecone.grpc import PineconeGRPC as PineconeClient
from pinecone import ServerlessSpec

# ----------------
# CONFIG (edit only these if needed)
# ----------------
index_name = "medicalbot"     # your index
namespace = "medical"         # must match upsert & retrieval
pdf_folder = "../data"        # change to "../Data" if folder name is Data
question = "what is acne"     # test query

# ----------------
# Load .env (notebook in /research, .env in project root)
# ----------------
ENV_PATH = Path("..") / ".env"
load_dotenv(ENV_PATH, override=True)

api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    raise ValueError("PINECONE_API_KEY missing. Put it in ../.env")

# Ensure all libs see it
os.environ["PINECONE_API_KEY"] = api_key

print("✅ .env:", ENV_PATH.resolve(), "| exists:", ENV_PATH.exists())
print("✅ pdf_folder:", Path(pdf_folder).resolve(), "| exists:", Path(pdf_folder).exists())

# ----------------
# Extract Data From the PDF File
# ----------------
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

# ----------------
# Split the Data into Text Chunks
# ----------------
def text_split(extracted_data):
    text_chunks = []
    chunk_size = 500
    chunk_overlap = 20
    step = chunk_size - chunk_overlap

    for d in extracted_data:
        text = d.page_content or ""
        meta = d.metadata or {}
        for i in range(0, len(text), step):
            chunk = text[i:i + chunk_size]
            if chunk.strip():
                text_chunks.append(Document(page_content=chunk, metadata=meta))

    return text_chunks

# ----------------
# Download the Embeddings from Hugging Face
# ----------------
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

# ----------------
# RUN PIPELINE
# ----------------
extracted_data = load_pdf_file(data=pdf_folder)
print("Pages loaded:", len(extracted_data))

text_chunks = text_split(extracted_data)
print("Length of Text Chunks:", len(text_chunks))
print("Sample chunk:", text_chunks[0].page_content[:200] if text_chunks else "NO CHUNKS")

embeddings = download_hugging_face_embeddings()
test_vec = embeddings.embed_query("Hello world")
print("Embedding dimension:", len(test_vec))

# ----------------
# Ensure Pinecone index exists (THIS project)
# ----------------
pc = PineconeClient(api_key=api_key)
existing_indexes = [i["name"] for i in pc.list_indexes()]
print("Indexes in this Pinecone project:", existing_indexes)

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("✅ Created index:", index_name)
else:
    print("✅ Index exists:", index_name)

# ----------------
# Upsert to Pinecone (LangChain)
# ----------------
docsearch = Pinecone.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name,
    namespace=namespace
)
print("✅ Upsert complete:", index_name, "| namespace:", namespace)

# ----------------
# Truth check: index stats
# ----------------
index = pc.Index(index_name)
stats = index.describe_index_stats()
print("✅ Index stats:", stats)

# ----------------
# Load existing + Retriever + Query
# ----------------
docsearch2 = Pinecone.from_existing_index(
    index_name=index_name,
    embedding=embeddings,
    namespace=namespace
)

retriever = docsearch2.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke(question)

print("Question:", question)
print("Docs returned:", len(docs))

for i, d in enumerate(docs, 1):
    print(f"\n--- DOC {i} ---")
    print(d.page_content[:400])

# If nothing returned, run a guaranteed-match query from your own data
if len(docs) == 0 and len(text_chunks) > 0:
    guaranteed_q = text_chunks[0].page_content[:60]
    docs2 = retriever.invoke(guaranteed_q)
    print("\n⚠️ No docs for your question. Testing guaranteed-match query...")
    print("Guaranteed query:", guaranteed_q)
    print("Docs returned:", len(docs2))
    for i, d in enumerate(docs2, 1):
        print(f"\n--- MATCH DOC {i} ---")
        print(d.page_content[:400])


Pages loaded: 637
Length of Text Chunks: 5799
Sample chunk: The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
Embedding dimension: 384
Indexes in this Pinecone project: ['medicalbot']
✅ Index exists: medicalbot
✅ Upsert complete: medicalbot | namespace: medical
✅ Index stats: {'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'medical': {'vector_count': 25196}},
 'total_vector_count': 25196}
Question: what is acne
Docs returned: 5

--- DOC 1 ---
see Heartburn
Acidosis see Respiratory acidosis; Renal
tubular acidosis; Metabolic acidosis
Acne
Definition
Acne is a common skin disease characterized by
pimples on the face, chest, and back. It occurs when the
pores of the skin become clogged with oil, dead skin
cells, and bacteria.
Description
Acne vulgaris, the medical term for common acne, is
the most common skin disease. It affects nearly 17

--- DOC 2 ---
see Heartburn
Acidosis see Respiratory acidosis; Renal
tubular acidosis; Metabolic acidosis
Acne
Definition
Acne is a common sk

In [9]:
from dotenv import load_dotenv
from pathlib import Path
load_dotenv(Path("..") / ".env", override=True)

import os
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))


OpenAI key loaded: True


In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.4,
    max_tokens=500
)


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1) Prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# 2) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4, max_tokens=500)

# 3) Retriever
retriever = docsearch.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 4) RAG chain (IMPORTANT: retriever must receive a STRING, not dict)
rag_chain = (
    {
        "context": (lambda x: x["input"]) | retriever | format_docs,
        "input": lambda x: x["input"],
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 5) Invoke
response = {"answer": rag_chain.invoke({"input": "What is Acne?"})}
print(response["answer"])


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}